In [ ]:
import json

# Load the tiddlers JSON from the file
with open("tiddlers.json", "r", encoding="utf-8") as f:
    nodesAll = json.load(f) # convierte el texto (str) en diccionario de python

# Crear el diccionario de todos los nodos
nodes_dict = {}

# Para cada elemento de la lista, asignar como nombre del nodo el elemento (atributo) "title"
for node in nodesAll:
    nomNode = node["title"]
    
    # Crear un subdiccionario de atributos para cada nodo (sin el elemento "title")
    nodeAttributes = {}
    # convertir los elementos de la lista en clave:valor de un diccionario
    for key, value in node.items():
        if key != "title":
            nodeAttributes[key] = value
            
    # Asignar al nombre del nodo (nomNode) el subdiccionario de los atributos
    nodes_dict[nomNode] = nodeAttributes
print("Nodes Dictionary: ")
print(nodes_dict)

edges_dict = {}
# Para cada elemento de la lista, tomar el elemento "tmap.edges" 
for node in nodesAll:
    nomNode = node["title"]
    edges_json = node.get("tmap.edges", "")
    
    if edges_json:  # Verifica que no esté vacío
        print(f"{nomNode} has edges.")
        edges_data = json.loads(edges_json) # convierte el texto(str) en diccionario de python
        # Crear un subdiccionario de relaciones para cada edge
        edgeRelation = {}
        for edge_id, edge_info in edges_data.items():
            edgeRelation[edge_id] = edge_info
        # Asignar al nombre del nodo (nomNode) el subdiccionario de los edges
        edges_dict[nomNode] = edgeRelation
    else:
        print(f"{nomNode} has NO edges.")

print("Edges Dictionary: ")
print(edges_dict)    

# Crear lista de almacenamiento de instrucciones para insertar los nodos en TypeDB
typedb_statements = []   


# Contadores por tipo de entidad
counters = {
    "cost": 1,
    "infrastructure": 1,
    "service": 1, 
    "revenue": 1
}

# Create entities for each node
for node_id, node in nodes_dict.items():
    entityType = node.get("tags", "").split()[0].lower()

    # atributos
    approximateValueEuro = node.get("approximateValueEuro", "")
    Comments = node.get("Comments", "")
    concernedDomain = node.get("concernedDomain", "")
    constructionDate = node.get("constructionDate", "")
    estimationPrecision = node.get("estimationPrecision", "")
    expectedLifeTime = node.get("expectedLifeTime", "")
    frequency = node.get("frequency", "")
    function = node.get("function", "")
    necessary = node.get("necessary", "")
    planned = node.get("planned", "")
    referencePeriod = node.get("referencePeriod", "")
    regulated = node.get("regulated", "")
    statusDevelopment = node.get("statusDevelopment", "")
    tangible = node.get("tangible", "")

    if entityType == 'cost':
        nameCost = node_id
        variable = f"$cost{counters['cost']}"
        statement = (
            f'insert \n\t{variable} isa {entityType}, has name "{nameCost}";\n'
            f'\t{variable} isa {entityType}, has approximateValueEuro {approximateValueEuro};\n'
            f'\t{variable} isa {entityType}, has estimationPrecision {estimationPrecision};\n'
            f'\t{variable} isa {entityType}, has frequency "{frequency}";\n'
            f'\t{variable} isa {entityType}, has planned "{planned}";\n'
            f'\t{variable} isa {entityType}, has referencePeriod "{referencePeriod}";\n'
            f'\t{variable} isa {entityType}, has Comments "{Comments}";\n'
        )
        typedb_statements.append(statement)

        print(statement)
        counters['cost'] += 1  # Incrementar contador de 'cost'
        
    elif entityType == 'infrastructure':
        nameInfra = node_id
        variable = f"$infra{counters['infrastructure']}"
        statement = (
            f'insert \n\t{variable} isa {entityType}, has name "{nameInfra}";\n'
            f'\t{variable} isa {entityType}, has concernedDomain "{concernedDomain}";\n'
            f'\t{variable} isa {entityType}, has function "{function}";\n'
            f'\t{variable} isa {entityType}, has regulated "{regulated}";\n'
            f'\t{variable} isa {entityType}, has necessary "{necessary}";\n'
            f'\t{variable} isa {entityType}, has constructionDate "{constructionDate}";\n'
            f'\t{variable} isa {entityType}, has Comments "{Comments}";\n'
        )
        typedb_statements.append(statement)
        print(statement)
        # Create relation structure 
            #relation_dict={}   
          
        for nomNode, edges in edges_dict.items():
            if nomNode ==  nameInfra:
                for edge_id, edge_info in edges.items():
                    edgeType = edge_info.get("type", "")[:-1]
                    relation = f"insert\n \t" + edgeType + " (" + edgeType + "ed: "+ edge_info.get("to", "") + ", " + edgeType + ": " + variable + ");"
                    typedb_statements.append(relation) 
                    print(relation)
        counters['infrastructure'] += 1  # Incrementar contador de 'infrastructure'

    elif entityType == 'service':
        nameServ = node_id
        variable = f"$serv{counters['service']}"
        statement = (
            f'insert \n\t{variable} isa {entityType}, has name "{nameServ}";\n'
            f'\t{variable} isa {entityType}, has concernedDomain "{concernedDomain}";\n'
            f'\t{variable} isa {entityType}, has tangible "{tangible}";\n'
            f'\t{variable} isa {entityType}, has Comments "{Comments}";\n'
        )
        typedb_statements.append(statement)
        print(statement)
        counters['service'] += 1 
    else:
        nameRev = node_id
        variable = f"$rev{counters['revenue']}"
        statement = (
            f'insert \n\t{variable} isa {entityType}, has name "{nameRev}";\n'
            f'\t{variable} isa {entityType}, has approximateValueEuro {approximateValueEuro};\n'
            f'\t{variable} isa {entityType}, has estimationPrecision {estimationPrecision};\n'
            f'\t{variable} isa {entityType}, has frequency "{frequency}";\n'
            f'\t{variable} isa {entityType}, has tangible "{tangible}";\n'
            f'\t{variable} isa {entityType}, has referencePeriod "{referencePeriod}";\n'
            f'\t{variable} isa {entityType}, has Comments "{Comments}";\n'
        )
        typedb_statements.append(statement)
        print(statement)
        counters['revenue'] += 1  # Incrementar contador de 'revenue'

# Create relation structure 
#relation_dict={}      
for nomNode, edges in edges_dict.items():
    for edge_id, edge_info in edges.items():
        relation = f"insert\n \t" + edge_info.get("type", "")[:-1] + " (" + edge_info.get("type", "")[:-1] + "ed: "+ edge_info.get("to", "") + ", " + edge_info.get("type", "") + ": " + nomNode + ");"
        typedb_statements.append(relation) 
        print(relation)
 
with open("typedb_inserts.gql", "w", encoding="utf-8") as output_file:
    for stmt in typedb_statements:
        output_file.write(stmt + "\n")


Nodes Dictionary: 
{'GeothermalWell': {'created': '20251009085223375', 'tags': 'Infrastructure Heat Water', 'modified': '20251009085322911', 'concernedDomain': 'water manegement', 'function': 'water capture process', 'regulated': 'Yes', 'necessary': 'Yes', 'constructionDate': '1997', 'expectedLifeTime': '40', 'statusDevelopment': 'executed', 'tmap.id': '353b8679-d8dc-4ae8-95f5-d5aa95ea30f2', 'text': '', 'tmap.edges': '{"79a55657-2e15-40be-8b01-35adf39a584d":{"to":"782b22bd-13ba-4a69-8fbb-54d5af6c76d5","type":"needs"},"241b3634-7065-4e44-a940-fc4807be65ef":{"to":"b7ab591c-c26d-4155-92cb-cad6a8e8870d","type":"provides"},"5efbb6b0-f838-4fd1-b664-555c7132f69a":{"to":"b76d71b7-6166-4406-8816-d61b0780668b","type":"provides"}}'}, 'InfrastructureManagement': {'created': '20251009085425445', 'tags': 'Cost Heat Water', 'modified': '20251009085504420', 'Comments': 'Complete system, including the well, 2 pumps, collector, pipe system and installation, the total cost is approximatley €100,000', 'ap